In [ ]:
from pathlib import Path

import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.dataset as ds
import pyarrow.parquet as pq


# ============================================================
# Settings
# ============================================================

indir = Path("antares_data")
outdir = Path("antares_data_from_20260527")

alert_outdir = outdir / "alerts"
locus_outdir = outdir / "loci"

alert_outdir.mkdir(parents=True, exist_ok=True)
locus_outdir.mkdir(parents=True, exist_ok=True)

# 2026-05-27 00:00:00 UTC
mjd_min = 61187.0


# ============================================================
# Read alert dataset
# ============================================================

alerts_ds = ds.dataset(
    indir / "alerts",
    format="parquet",
)

# Keep alerts from May 27 onward
scanner = alerts_ds.scanner(
    filter=ds.field("mjd") >= mjd_min
)


# ============================================================
# Write filtered alerts
#
# Stream by batches so we do not need to load all
# alert_properties into memory at once.
# ============================================================

alert_file = alert_outdir / "alerts_00000.parquet"

writer = None
kept_locus_ids = set()
n_alerts_kept = 0

for batch in scanner.to_batches():

    if batch.num_rows == 0:
        continue

    # Record locus IDs that still have alerts
    i = batch.schema.get_field_index("locus_id")

    kept_locus_ids.update(
        batch.column(i).to_pylist()
    )

    # Open writer when first batch arrives
    if writer is None:

        writer = pq.ParquetWriter(
            alert_file,
            batch.schema,
            compression="zstd",
        )

    writer.write_batch(batch)

    n_alerts_kept += batch.num_rows


if writer is not None:
    writer.close()


# ============================================================
# Filter loci
#
# Keep only loci having at least one retained alert
# ============================================================

loci_ds = ds.dataset(
    indir / "loci",
    format="parquet",
)

# Only ~10k rows, so loading the locus table is small
loci_table = loci_ds.to_table()

keep_ids = pa.array(
    list(kept_locus_ids),
    type=pa.string(),
)

mask = pc.is_in(
    loci_table["locus_id"],
    value_set=keep_ids,
)

filtered_loci = loci_table.filter(mask)


# ============================================================
# Write filtered loci
# ============================================================

pq.write_table(
    filtered_loci,
    locus_outdir / "loci_00000.parquet",
    compression="zstd",
)


# ============================================================
# Summary
# ============================================================

n_alerts_original = alerts_ds.count_rows()
n_loci_original = loci_ds.count_rows()

print("Finished")
print()
print(f"Original alerts:  {n_alerts_original:,}")
print(f"Filtered alerts:  {n_alerts_kept:,}")
print(f"Alerts removed:   {n_alerts_original - n_alerts_kept:,}")
print()
print(f"Original loci:    {n_loci_original:,}")
print(f"Filtered loci:    {filtered_loci.num_rows:,}")
print(f"Loci removed:     {n_loci_original - filtered_loci.num_rows:,}")

In [ ]:
import pyarrow.dataset as ds
import pandas as pd
import matplotlib.pyplot as plt


alerts_ds = ds.dataset(
    "antares_data_from_20260527/alerts",
    format="parquet",
)

table = alerts_ds.to_table(
    columns=["locus_id"]
)

locus_ids = table["locus_id"].to_pylist()

counts = (
    pd.Series(locus_ids)
    .value_counts()
)

print(counts.describe())


fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(
    counts.values,
    bins=30,
)

ax.set_xlabel("Number of retained alerts per locus")
ax.set_ylabel("Number of loci")

fig.tight_layout()
plt.show()

In [ ]:
print("Minimum alerts per locus:", counts.min())
print("Maximum alerts per locus:", counts.max())

In [ ]:
import pyarrow.dataset as ds
import pandas as pd

loci_ds = ds.dataset(
    "antares_data/loci",
    format="parquet",
)

locus_ids = loci_ds.to_table(
    columns=["locus_id"]
)["locus_id"].to_pylist()

s = pd.Series(locus_ids)

print("Total locus rows:", len(s))
print("Unique locus IDs:", s.nunique())
print("Duplicated rows:", s.duplicated().sum())

In [ ]:
counts_loci = s.value_counts()

print(
    counts_loci[counts_loci > 1]
    .sort_values(ascending=False)
    .head(20)
)

In [ ]:
from pathlib import Path

files = sorted(Path("antares_data/loci").glob("*.parquet"))

print("Number of locus files:", len(files))

for f in files:
    print(f.name)

In [ ]:
import pyarrow.dataset as ds
import pandas as pd
from pathlib import Path

files = sorted(Path("antares_data/loci").glob("*.parquet"))

print("Files:")
for f in files:
    print(f.name)

loci = ds.dataset(
    "antares_data/loci",
    format="parquet",
).to_table(
    columns=["locus_id"]
).to_pandas()

print()
print("Rows:", len(loci))
print("Unique:", loci["locus_id"].nunique())
print("Duplicates:", loci["locus_id"].duplicated().sum())